# Position Risk Controls - FX Pairs

The two stages before this one decided what to hold and how much. A position risk rule decides
when to stop holding it. That makes this the first stage whose effect depends on the price path
inside a holding period rather than on its endpoints: a position that ends the period down four
percent and one that dipped nine percent on the way to ending down four are indistinguishable
to every earlier stage, and a stop separates them.

The rules compared here are declared in `config/setup.yaml`, not invented in the notebook: four
fixed stop-loss levels and five trailing stops. Portfolio-level controls - a drawdown cap or a
daily loss limit applied to the book as a whole - are absent for this case study, so nothing in
this stage restrains the aggregate. Each position is governed on its own.

This notebook selects one validation strategy per label from the equal-weight and allocation
populations, then changes only its predeclared position-risk rule. Cost-sensitivity results are
excluded from selection, and the reason is a selection defect rather than tidiness: a cost sweep
re-prices one strategy under several assumptions, so a ranking that included those rows would
advance whichever strategy had been measured under the kindest cost model. The comparison would
read as a difference between strategies and be a difference between assumptions.

**Learning objectives**

- Select one parent strategy from an immutable validation cohort.
- Compare declared position controls while preserving upstream identity.
- Freeze the complete risk population before final strategy selection.
- See why the count of variants tried belongs beside any improvement they produce.

**Book reference**: Chapter 19

**Prerequisite**: `14_portfolio_management`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 部位風險控制——外匯貨幣對

前兩個階段決定持有什麼，以及持有多少。部位風險規則則決定何時停止持有。因此，這是第一個效果取決於持有期間內價格路徑、而非只取決於起終點的階段：某部位期末下跌百分之四，與另一部位中途曾下跌百分之九、最終也下跌百分之四，在此前所有階段看來並無差別；停損規則則能區分兩者。

這裡比較的規則宣告於 `config/setup.yaml`，不是在 Notebook 中臨時發明：包含四種固定停損水準與五種追蹤停損。本案例研究未納入投資組合層級的控制措施——例如對整體帳簿套用的回撤上限或每日虧損限制——因此本階段不會限制整體風險。每個部位都各自受規則約束。

本 Notebook 從等權重與配置模型集合中，為每種標籤選出一個驗證期策略，之後只改變其預先宣告的部位風險規則。成本敏感度結果會排除於選擇之外；原因是選擇缺陷，而非僅為整齊：成本參數掃描會在多種假設下重新為同一策略定價，因此若將這些資料列納入排序，進入下一階段的就會是剛好在最寬鬆成本模型下衡量的策略。比較表面上像是策略間差異，實際上卻是不同假設造成的差異。

**學習目標**

- 從不可變的驗證期候選群組中選出一個母策略。
- 在保留上游識別資訊的情況下，比較預先宣告的部位控制措施。
- 在最終策略選擇前，固定完整的風險模型集合。
- 了解為何嘗試的變體數量，應與它們帶來的改善一同列示。

**書籍參考**：第 19 章

**先備條件**：`14_portfolio_management`。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""Run the declared FX position-risk controls for each label."""

from copy import deepcopy
from typing import Any

import polars as pl
import yaml

from case_studies.research import (
    BacktestResult,
    CandidateSet,
    OfficialPopulation,
    Result,
    candidate_set_supersedes,
    open_study,
    plan_backtests,
    population_supersedes,
    research_name,
    reuse_disclosure,
    run_backtests,
    superseded_members,
)
from case_studies.utils.backtest_presets import EngineBacktestConfig
from case_studies.utils.strategy_analysis import selectable_validation_candidates
from case_studies.utils.sweep_config import (
    get_portfolio_risk_controls,
    get_position_risk_controls,
    get_top_n_predictions,
)
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
LABEL = ""
SPLIT = "validation"
TOP_K = 0
TOP_N_PREDICTIONS = None
MAX_RISK_VARIANTS = 0
# How many parents per label the overlay grid sits on. `None` reads
# `backtest.sweep.top_n_predictions.risk_overlay`, which every case study declares as 1, and one
# is narrow on purpose: an overlay is a second search over the same validation folds, so the
# question the book asks is whether a control improves the configuration the funnel already
# chose. Until 2026-09-20 that 1 was a literal `min(eligible, ...)` below rather than a number
# read from the declaration, which left this case study unable to answer at any other width
# while four others could. A wider run changes the member list of every name published here, so
# it is narrowing and widening alike that need `POPULATION_NAME`, asserted below.
TOP_N_COMBOS = None
SEED = 42
RUN_SWEEP = True
FORCE_REBACKTEST = False
POPULATION_NAME = ""
# `POPULATION_NAME` scopes what this notebook *writes*. The equal-weight baselines it reads are
# written by `13_backtest` and carry the canonical name, so a scoped run that inherits the scope
# on that lookup asks for a population no run ever wrote and stops with "resolved to 0 current
# identities among 0 snapshots". Empty string reads the canonical baselines while the outputs
# stay scoped, which is the shape a partial re-run needs; None means "same scope as the outputs".
# `14_portfolio_management` gained the same parameter in #1109 and this is its sibling.
BASELINE_POPULATION_NAME: str | None = None
# `df20d72ab319` was the tip of `fx_pairs:risk-overlay-backtests` when it was written, and
# `create` accepts the tip and nothing else, so the literal was correct exactly until this
# notebook next published. `"live"` names the lineage and is resolved against it at run time.
SUPERSEDES_RISK_BACKTESTS: str = "live"
# A candidate set is immutable under its name, exactly as a population is, so a rebuilt upstream
# generation has to name the set it replaces. Keyed by the full set name because that is what the
# refusal prints: pasting back the name it names is the obvious thing to try, and it has to work.
# Resolved through `candidate_set_supersedes` rather than passed straight to `create`, because a
# reader's clean clone has no generation to supersede and `create` refuses a first version that
# claims to replace one.
#
# `"live"` names the lineage rather than a generation of it, which is what stops these going
# stale again. Three of the four hashes it replaces were already dead: the three
# `pre-risk-strategies` sets moved on 2026-09-09 (`d966caa61faf` -> `669d50f0f525`,
# `fde7af05fff6` -> `93b56a9dfb42`, `ff269dc95622` -> `f70630285818`), so each named the
# generation its own successor had replaced and the next membership move here would have been
# refused at the freeze, after the fit. `bf21ae4c9070` was still the head of
# `fx_pairs:holdout-candidates` and would have gone the same way on the next publish.
# See `case_studies.research.population.SUPERSEDES_LIVE`.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "fx_pairs:fwd_ret_1d:pre-risk-strategies": "live",
    "fx_pairs:fwd_ret_5d:pre-risk-strategies": "live",
    "fx_pairs:fwd_ret_21d:pre-risk-strategies": "live",
    "fx_pairs:holdout-candidates": "live",
}

## Select the parent strategy

Production uses the same signal-plus-allocation candidate cohort as the cost notebook. Preview
mode resolves one deterministic allocation result from the reduced prediction catalog.

One parent per label, chosen by best validation backtest Sharpe from the sealed pre-risk cohort.
Sharing the cohort with the cost notebook is deliberate: both stages branch from the same
strategy, so their results are siblings of one parent and can be read against each other. If
each stage picked its own parent, a cost result and a risk result for the same label would
describe two different strategies and the comparison between them would mean nothing.

The cohort is sealed before it is ranked, which is what makes "best" a statement about a fixed
set. A cohort that could still gain members after the selection would let a later run change
which strategy this notebook chose, retroactively, with no record that it had changed.

The `identity_status == "current"` filter below carries the same trap `13_backtest` documents:
it names the schema version a row was written under, not whether its producer still publishes
it. A refit leaves the replaced generation in the registry, complete and current, so the filter
alone would let a retired prediction set into the cohort - and nothing would fail, because a
retired prediction still produces a correct backtest. `superseded_members` reads the lineage,
which is where the retirement is recorded.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 選擇母策略

正式執行使用與成本 Notebook 相同的訊號加配置候選群組。預覽模式則從縮減的預測目錄中，以可重現方式解析出一項配置結果。

每種標籤各有一個母策略，從風險調整前已封存的候選群組中，依驗證期回測夏普比率最高者選出。刻意與成本 Notebook 共用候選群組：兩個階段都從同一策略分支，因此結果都是同一母策略的平行變體，可以彼此對照。若各階段自行挑選母策略，同一標籤的成本結果與風險結果就會描述兩種不同策略，兩者之間的比較也毫無意義。

候選群組會在排序前封存，因此「最佳」是對固定集合的陳述。若選擇後仍能新增成員，之後某次執行便可能追溯改變本 Notebook 選出的策略，卻不留下任何變更紀錄。

下方的 `identity_status == "current"` 篩選條件，具有 `13_backtest` 所記載的相同陷阱：它標示資料列寫入時的架構版本，而不是產生端是否仍發布該資料列。重新擬合後，被取代的一代仍留在登錄系統，保持完整且標記為目前可用，因此單靠此篩選條件，就會讓已退役的預測集進入候選群組——也不會失敗，因為退役預測仍能產生正確回測。`superseded_members` 會讀取版本沿革，退役資訊才記錄在那裡。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
set_global_seeds(SEED)
universe_symbols = yaml.safe_load(
    (get_case_study_dir(CASE_STUDY_ID) / "config" / "setup.yaml").read_text()
)["universe"]["symbols"]
n_assets = len(universe_symbols)
if SPLIT != "validation":
    raise ValueError("risk comparison uses validation backtests")
if FORCE_REBACKTEST:
    raise ValueError("identical complete backtests are reused by identity")
if not RUN_SWEEP:
    raise ValueError("set RUN_SWEEP=True to execute the visible risk request")

study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
# The execution tier decides which registry namespace this run reads and writes;
# the reduction knobs decide only how much of it is covered. Inferring the tier
# from the knobs conflated the two, so any reduced run went looking for preview
# predictions - and a reduced run over a canonical upstream, which is what the
# test suite exercises, then resolved no rows at all.
include_preview = EXECUTION_TIER == "preview"


def _resolve_baseline_scope(output_scope: str, input_scope: str | None) -> str:
    return output_scope if input_scope is None else input_scope


baseline_population_name = _resolve_baseline_scope(POPULATION_NAME, BASELINE_POPULATION_NAME)

# The tier decides the namespace, so a canonical run may legitimately be narrowed -
# but a narrowed run declares a different set of members than the canonical
# population does, and a population is immutable once written. Such a run must
# publish under its own name rather than register a partial snapshot of the risk overlay sweep
# under the canonical one.
if (
    (
        TOP_K
        or TOP_N_PREDICTIONS is not None
        or MAX_RISK_VARIANTS
        or LABEL
        or TOP_N_COMBOS is not None
    )
    and not include_preview
    and not POPULATION_NAME
):
    raise ValueError(
        "this run does not sweep the declared risk overlay population, so it cannot publish "
        "the canonical one; pass POPULATION_NAME to give it its own"
    )
if TOP_N_COMBOS is None:
    TOP_N_COMBOS = get_top_n_predictions(CASE_STUDY_ID, "risk_overlay")
if TOP_N_COMBOS < 1:
    raise ValueError("the risk overlay needs at least one parent per label")
catalog = study.predictions.table(include_preview=include_preview).filter(
    (pl.col("identity_status") == "current")
    & (pl.col("split") == SPLIT)
    & pl.col("complete")
    & (pl.col("execution_tier") == ("preview" if include_preview else "canonical"))
)
# `identity_status` is the schema version a row was written under, not a statement about which
# generation its producer publishes. A model notebook that refits leaves the generation it
# replaced in the registry, complete and current, so this filter alone would carry a retired
# prediction set into the sweep. `superseded_members` reads the lineage instead - see
# `13_backtest`, which drops the same set before it freezes the baseline population.
# `SUPERSEDES_RISK_BACKTESTS` is the sentinel `"live"`, so it names the lineage this run
# publishes under and `population_supersedes` resolves the generation in force at write time.
# It resolves to nothing until that name has a first generation, which is also what a reader's
# clean clone sees; after that, an upstream refit changes this population's member list and the
# registry refuses the write without the tip. `13_backtest` states the reasoning once.
retired = superseded_members(study, member_kind="prediction")
if retired:
    catalog = catalog.filter(~pl.col("prediction_hash").is_in(list(retired)))
if LABEL:
    catalog = catalog.filter(pl.col("label") == LABEL)
if TOP_N_PREDICTIONS is not None:
    catalog = catalog.sort("label", "family", "config_name", "checkpoint_value").head(
        TOP_N_PREDICTIONS
    )
if catalog.is_empty():
    raise RuntimeError("risk comparison resolved no complete prediction rows")


def _open_backtests(population: OfficialPopulation) -> list[BacktestResult]:
    population.require_complete()
    opened = [Result.open(study, value) for value in population.members]
    if any(not isinstance(result, BacktestResult) for result in opened):
        raise TypeError(f"population {population.name!r} contains a non-backtest result")
    return [result for result in opened if isinstance(result, BacktestResult)]


def _label(result: BacktestResult) -> str:
    return str(result.lineage()["training_spec"]["label"])


def _registered_preview_allocations() -> pl.DataFrame:
    """The allocation backtests an upstream preview registered.

    One predicate, read once. Stating it twice - once to decide which labels are covered
    and once to pick a label's leader - makes the two required to agree forever: tighten
    one and the other admits a label whose backtests it then rejects, which is exactly the
    "no preview allocation backtests are registered" failure this exists to prevent.
    """
    return study.backtests.table(include_preview=True).filter(
        (pl.col("stage") == "allocation")
        & (pl.col("execution_tier") == "preview")
        & (pl.col("identity_status") == "current")
        & pl.col("complete")
    )


def _preview_leader(rows: pl.DataFrame, registered_allocations: pl.DataFrame) -> BacktestResult:
    """One allocation backtest for this label, read from what 14 registered.

    Rebuilding the identity restated two of the upstream run's choices - its `top_k` and
    which allocator came first - from this notebook's own defaults. A preview reduces
    each notebook independently, so those are guesses about another run's parameters,
    and a guess that is wrong looks for a hash nothing wrote instead of reporting the
    disagreement.
    """
    registered = registered_allocations.filter(
        pl.col("prediction_hash").is_in(rows.get_column("prediction_hash").implode())
    )
    if registered.is_empty():
        raise RuntimeError(
            "no preview allocation backtests are registered for this prediction "
            "catalog; run 14_portfolio_management at the same reduction first"
        )
    row = registered.sort("family", "config_name", "checkpoint_value", "backtest_hash").row(
        0, named=True
    )
    result = Result.open(study, row["backtest_hash"], include_preview=True)
    if not isinstance(result, BacktestResult) or not result.complete:
        raise RuntimeError("the deterministic preview allocation result is not complete")
    return result


selected_by_label: dict[str, list[BacktestResult]] = {}
candidate_sets: dict[str, CandidateSet] = {}
if include_preview:
    # The labels come from what the upstream preview registered, the same rule the
    # canonical branch below follows. Enumerating this notebook's own catalog asks
    # _preview_leader for labels 14_portfolio_management was configured not to allocate,
    # and it raises "no preview allocation backtests are registered" for each - reporting
    # a reduction the run was told to make as a missing upstream.
    registered_allocations = _registered_preview_allocations()
    covered = catalog.filter(
        pl.col("prediction_hash").is_in(
            registered_allocations.get_column("prediction_hash").implode()
        )
    )
    if covered.is_empty():
        raise RuntimeError(
            "no preview allocation backtests cover this prediction catalog; "
            "run 14_portfolio_management at the same reduction first"
        )
    for label in sorted(covered.get_column("label").unique()):
        selected_by_label[label] = [
            _preview_leader(covered.filter(pl.col("label") == label), registered_allocations)
        ]
else:
    baselines = _open_backtests(
        OfficialPopulation.one(
            study,
            name=research_name(
                CASE_STUDY_ID, "equal-weight-baselines", scope=baseline_population_name
            ),
        )
    )
    allocations = _open_backtests(
        OfficialPopulation.one(
            study,
            name=research_name(CASE_STUDY_ID, "allocation-backtests", scope=POPULATION_NAME),
        )
    )
    # The labels come from the upstream populations this run resolved, not from this
    # notebook's own catalog. A narrowed upstream covers fewer labels than the catalog
    # holds, and rebuilding the list locally reproduces the upstream narrowing by
    # convention. Unscoped, the run publishes canonical names and the two must agree.
    upstream = [*baselines, *allocations]
    upstream_labels = sorted({_label(result) for result in upstream})
    if not upstream_labels:
        raise RuntimeError("the upstream populations carry no labels")
    if not POPULATION_NAME and upstream_labels != sorted(catalog.get_column("label").unique()):
        raise RuntimeError(
            "the canonical upstream populations do not cover every label in the catalog: "
            f"upstream {upstream_labels}, "
            f"catalog {sorted(catalog.get_column('label').unique())}"
        )
    # Eligibility and order both come from `selectable_validation_candidates`, the function
    # `resolve_solvent_carrier` ranks. Re-deriving them here swept the risk variants over a
    # strategy the case study never publishes: the populations above are read whole, and nothing
    # applies the retired-prediction test to them, so a backtest whose prediction a later refit
    # superseded still won on raw Sharpe. `16_costs` carried the same defect.
    _eligible_order = {
        row["backtest_hash"]: position
        for position, row in enumerate(
            selectable_validation_candidates(CASE_STUDY_ID, labels=[LABEL] if LABEL else None)
        )
    }
    for label in upstream_labels:
        members = [result for result in upstream if _label(result) == label]
        eligible = [result for result in members if result.hash in _eligible_order]
        if not eligible:
            raise RuntimeError(
                f"none of the {len(members)} upstream backtests for {label} is selectable: "
                "every one is retired on the backtest or the prediction side, or belongs to no "
                "population its producer publishes. Re-run the validation stages rather than "
                "sweeping risk controls over a strategy nothing reports."
            )
        _set_name = research_name(
            CASE_STUDY_ID, f"{label}:pre-risk-strategies", scope=POPULATION_NAME
        )
        # The frozen set records the field the selection actually saw, so it holds the
        # selectable members and not every row the three populations list.
        candidates = CandidateSet.create(
            study,
            name=_set_name,
            members=eligible,
            supersedes=candidate_set_supersedes(
                study, name=_set_name, declared=SUPERSEDES_CANDIDATE_SETS.get(_set_name)
            ),
        )
        candidate_sets[label] = candidates
        leaders = sorted(eligible, key=lambda result: _eligible_order[result.hash])[:TOP_N_COMBOS]
        if not all(isinstance(leader, BacktestResult) for leader in leaders):
            raise TypeError("strategy selection did not return a backtest")
        selected_by_label[label] = leaders

pl.DataFrame(
    [
        {
            "label": label,
            "backtest_hash": result.hash,
            "prediction_hash": result.registry_record()["prediction_hash"],
            "stage": result.registry_record()["stage"],
        }
        for label, results in selected_by_label.items()
        for result in results
    ]
)

label,backtest_hash,prediction_hash,stage
str,str,str,str
"""fwd_ret_1d""","""02df24286767""","""0eb73fea6afa""","""signal"""
"""fwd_ret_21d""","""0c8d095371b6""","""b839846c1759""","""allocation"""
"""fwd_ret_5d""","""fc2c1d73880f""","""93c502488088""","""signal"""


## Plan and freeze exact risk siblings

Every position rule is declared in `config/setup.yaml`. Portfolio-level controls are absent for
this case study. The identity audit removes only the risk block and chapter label; the prediction,
signal, allocation, configured costs, and execution contract must remain unchanged. Production
freezes every expected risk identity before the first backtest is written.

The audit is what makes each result a sibling rather than another strategy. Two backtests that
differ in their stop and in nothing else can be subtracted; two that differ in their stop and
their cost model cannot, and no field in the output would say which case you are looking at.
Removing exactly the risk block and the chapter label, then requiring the remainder to match,
is how that is established rather than assumed.

How a stop is evaluated is worth knowing before reading its results. `StopLoss` and
`TrailingStop` in `ml4t.backtest.risk` trigger on the bar's low or high where the price frame
supplies a range, and fall back to the close where it does not, so the same threshold is a
different rule depending on what the bars carry. The fill is a separate configured choice:
filling at the stop price, at the bar's extreme, or at the close give materially different
answers for one triggered stop, and the gap between them widens exactly in the volatile periods
a stop is meant to handle. A stop's measured benefit is therefore partly a statement about the
fill model, and that is a declared assumption rather than a property of the market.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 規劃並固定確切的風險變體

每項部位規則都宣告於 `config/setup.yaml`。本案例研究沒有投資組合層級的控制措施。識別資訊稽核只移除風險區塊與章節標籤；預測、訊號、配置、已設定成本及執行約定都必須保持不變。正式執行會在第一項回測寫入前，固定每個預期風險識別資訊。

這項稽核使每個結果成為同一策略的平行變體，而非另一種策略。兩項回測若只在停損規則不同，就可以相減比較；若停損規則與成本模型都不同，就不能這麼做，而輸出的欄位也不會告訴讀者屬於哪種情況。精確移除風險區塊與章節標籤，再要求其餘部分相符，才能確立這點，而非僅做假設。

閱讀結果前，也應了解停損如何評估。`StopLoss` 與 `TrailingStop` 在 `ml4t.backtest.risk` 中，於價格資料框提供價格區間時，會依該價格柱的最低價或最高價觸發；若沒有區間，則退回以收盤價判定。因此同一門檻會因價格柱所含資訊不同而成為不同規則。成交價格是另行設定的選項：在停損價格、價格柱極值或收盤價成交，對同一次停損觸發會給出顯著不同的結果；而在停損本欲處理的波動時期，差距恰好會擴大。因此，測得的停損效益，部分反映成交模型；這是明確宣告的假設，而非市場本身的性質。
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
position_controls = get_position_risk_controls(CASE_STUDY_ID)
if get_portfolio_risk_controls(CASE_STUDY_ID):
    raise RuntimeError("FX declares position controls only")
if MAX_RISK_VARIANTS:
    position_controls = position_controls[:MAX_RISK_VARIANTS]
if not position_controls:
    raise RuntimeError("the position-risk grid is empty")


def _risk_payload(control: dict[str, Any]) -> dict[str, Any]:
    if control["type"] == "time_exit":
        rule = {"type": control["type"], "bars": int(control["bars"])}
    else:
        rule = {"type": control["type"], "threshold": float(control["threshold"])}
    return {"name": control["name"], "position_rules": [rule]}


def _catalog_row(result: BacktestResult) -> pl.DataFrame:
    prediction_hash = result.registry_record()["prediction_hash"]
    row = catalog.filter(pl.col("prediction_hash") == prediction_hash)
    if row.height != 1:
        raise RuntimeError(f"prediction {prediction_hash} resolved to {row.height} catalog rows")
    return row


def _strategy_arguments(result: BacktestResult) -> dict[str, Any]:
    strategy = result.spec()["strategy"]
    return {
        "signal": deepcopy(strategy["signal"]),
        "allocation": deepcopy(strategy.get("allocation")),
        "execution_mode": strategy.get("rebalance", {}).get("mode"),
    }


def _non_risk_projection(spec: dict[str, Any]) -> dict[str, Any]:
    projected = deepcopy(spec)
    projected.pop("chapter", None)
    projected.pop("_runtime_backtest_config", None)
    projected.get("strategy", {}).pop("risk", None)
    metadata = projected.get("backtest_config", {}).get("metadata")
    if isinstance(metadata, dict):
        metadata.pop("chapter", None)
        # An absolute filesystem path, and already excluded from the identity hash by
        # `_HASH_EXCLUDED_METADATA` for that reason. Comparing it here makes the notebook
        # refuse its own siblings from any checkout but the one that registered the parents.
        metadata.pop("preset_path", None)
    # The parent allocation row was serialized by whatever engine version registered it and the
    # risk result by the installed one, so a field `BacktestConfig` has since gained is absent on
    # one side and present on the other while both describe the same strategy. `ml4t-backtest`
    # 0.1.3 to 0.1.6 added `account.lock_notional_update_mode` and
    # `position_sizing.share_rounding`, both previously implicit defaults the schema made
    # explicit: `NEAREST` is the rounding the engine already did, and `POSITION_LEGS` only bites
    # under a `lock_notional` short cash policy. Every fx_pairs parent predates them, so this
    # comparison reported a moved strategy field on two names for one behaviour.
    #
    # Round-tripping both sides through the installed schema states the comparison in one
    # vocabulary, so it answers what this notebook built rather than which engine wrote the row it
    # is compared against, and it covers the next added field without naming it.
    # `14_portfolio_management.py`, `16_costs.py` and `19_strategy_analysis.py` already do this;
    # this projection is the one that was missed. `ensure_backtest_spec` deliberately does NOT
    # round-trip, because there the result is hashed and a dropped unknown key would move an
    # identity; here it is compared and discarded. Metadata is merged back over the serialized
    # view because the dataclass pins a schema and drops keys it does not know.
    config = projected.get("backtest_config", {})
    if EngineBacktestConfig is not None and config:
        original_metadata = dict(metadata) if isinstance(metadata, dict) else {}
        rebuilt = EngineBacktestConfig.from_dict(config).to_dict()
        rebuilt_metadata = dict(rebuilt.get("metadata") or {})
        rebuilt_metadata.update(original_metadata)
        rebuilt["metadata"] = rebuilt_metadata
        projected["backtest_config"] = rebuilt
    return projected


risk_jobs = []
for label, selected_results in selected_by_label.items():
    for selected in selected_results:
        arguments = _strategy_arguments(selected)
        for control in position_controls:
            risk = _risk_payload(control)
            plan = plan_backtests(
                study,
                predictions=_catalog_row(selected),
                signal=arguments["signal"],
                allocation=arguments["allocation"],
                risk=risk,
                chapter="19",
                execution_mode=arguments["execution_mode"],
            )
            if len(plan.members) != 1:
                raise RuntimeError("a risk plan must contain exactly one backtest")
            risk_jobs.append(
                {
                    "label": label,
                    "selected": selected,
                    "arguments": arguments,
                    "risk": risk,
                    "risk_name": control["name"],
                    "backtest_hash": plan.expected_hashes[0],
                }
            )

planned_hashes = [job["backtest_hash"] for job in risk_jobs]
if len(planned_hashes) != len(set(planned_hashes)):
    raise RuntimeError("two planned risk requests collapse to the same identity")

risk_population = None
if not include_preview:
    risks_name = research_name(CASE_STUDY_ID, "risk-overlay-backtests", scope=POPULATION_NAME)
    risk_population = OfficialPopulation.create(
        study,
        name=risks_name,
        member_kind="backtest",
        members=planned_hashes,
        supersedes=population_supersedes(
            study, name=risks_name, declared=SUPERSEDES_RISK_BACKTESTS
        ),
    )
    print(f"Frozen expected risk population: {risk_population.hash}")

Frozen expected risk population: 9774f18e7c54


## Execute the frozen risk grid, and validate what was frozen

The population is validated in the cell that fills it: the expected set was written down before
the first member ran, and `require_complete` is what turns that declaration into a published
result.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行已固定的風險網格，並驗證固定集合

模型集合會在填入成員的儲存格中驗證：預期集合早在第一個成員執行前就已記錄，而 `require_complete` 會將宣告轉為已發布結果。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
# A sweep that recomputes everything and a sweep that recomputes nothing print the same summary
# unless the two are counted apart. `run_backtests` serves an identity that is already registered
# and complete instead of running it again, which is what makes a re-run affordable and what makes
# a bare member count say nothing about whether this run did any work.
#
# The runner already knows which it did and says so per member in `execution.diagnostics`, as
# `status` "reused" or "completed". Comparing against the registered hashes instead would be
# wrong in both directions: a registered-but-partial backtest is in that set, gets recomputed and
# would report as reused, and a preview re-run reads a table that excludes preview rows by default
# and would report every reused member as computed.
run_status: list[str] = []
risk_results: list[BacktestResult] = []
risk_rows = []
for job in risk_jobs:
    selected = job["selected"]
    arguments = job["arguments"]
    execution = run_backtests(
        study,
        predictions=_catalog_row(selected),
        signal=arguments["signal"],
        allocation=arguments["allocation"],
        risk=job["risk"],
        chapter="19",
        execution_mode=arguments["execution_mode"],
    )
    if len(execution.results) != 1:
        raise RuntimeError("a risk request must produce exactly one backtest")
    result = execution.results[0]
    if result.hash != job["backtest_hash"]:
        raise RuntimeError("a completed risk identity differs from the frozen plan")
    if _non_risk_projection(result.spec()) != _non_risk_projection(selected.spec()):
        raise RuntimeError("a risk sibling changed a non-risk strategy field")
    if result.registry_record()["stage"] != "risk_overlay" or not result.complete:
        raise RuntimeError("a risk result is incomplete or misclassified")
    risk_results.append(result)
    run_status.extend(entry["status"] for entry in execution.diagnostics)
    risk_rows.append(
        {
            "label": job["label"],
            "risk_name": job["risk_name"],
            "backtest_hash": result.hash,
            "prediction_hash": result.registry_record()["prediction_hash"],
        }
    )

served = run_status.count("reused")
print(
    f"Risk overlays: {reuse_disclosure(len(risk_results) - served, served)}, "
    f"{len(risk_results)} in the population"
)

if not include_preview:
    if risk_population is None:
        raise RuntimeError("the canonical risk population was not frozen before execution")
    risk_population.require_complete()
    print(f"Official risk population: {risk_population.hash}")

pl.DataFrame(risk_rows).sort("label", "risk_name")

Risk overlays: computed all 42, 42 in the population
Official risk population: 9774f18e7c54


label,risk_name,backtest_hash,prediction_hash
str,str,str,str
"""fwd_ret_1d""","""stop_loss_10pct""","""114fcd5c942a""","""0eb73fea6afa"""
"""fwd_ret_1d""","""stop_loss_15pct""","""2a9ca43f2292""","""0eb73fea6afa"""
"""fwd_ret_1d""","""stop_loss_3pct""","""8331c4e849ff""","""0eb73fea6afa"""
"""fwd_ret_1d""","""stop_loss_5pct""","""180b2a05a56a""","""0eb73fea6afa"""
"""fwd_ret_1d""","""time_exit_10""","""84ec7d832dd3""","""0eb73fea6afa"""
…,…,…,…
"""fwd_ret_5d""","""trailing_1pct""","""bc75435dd927""","""93c502488088"""
"""fwd_ret_5d""","""trailing_20pct""","""396feb7f1571""","""93c502488088"""
"""fwd_ret_5d""","""trailing_2pct""","""1bac3c7c0d25""","""93c502488088"""


## Freeze the set the holdout will choose from

Everything this case study has backtested on validation goes in: the equal-weight baselines, the
allocation variants, and the risk overlays. The comparison contract names the fields every member
must agree on, so a candidate fitted against different labels, features or folds cannot silently
join a set the holdout will pick from. Only identities are printed here; what the selection is
worth is `19_strategy_analysis`'s question.

The size of this set is the part that no table above reports and that every number below depends
on. Nine position rules were tried for each parent, on top of the allocators tried before them
and the configurations tried before those. The best member wins by some margin over the rest, and
part of that margin is simply the best of many draws on one validation period. Nothing in a
result marks it: the Sharpe, the drawdown and the hit rate of the winning stop are all correctly
computed for that stop, and a reader shown only the winner sees a strategy that looks better
rather than a maximum taken over a set whose size is not on the page. This is why the count of
members matters as much as the leader, why the set is frozen rather than pruned to the winner,
and why the holdout is spent once against the whole set rather than against the leader alone.

Freezing rather than pruning also keeps the negative results. A stop level that made things
worse is a finding about this strategy, and it is only visible while the members that lost are
still in the set beside the one that won.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 固定保留樣本將據以選擇的集合

本案例研究在驗證資料上回測過的所有項目都納入：等權重基準、配置變體與風險覆蓋規則。比較約定指明所有成員必須一致的欄位，因此對不同標籤、特徵或驗證折擬合的候選者，不能悄悄加入保留樣本將據以選擇的集合。這裡只列印識別資訊；選擇究竟有多少價值，是 `19_strategy_analysis` 要回答的問題。

這個集合的大小，是上方任何表格都未報告、卻影響下方每個數值的部分。每個母策略都嘗試了九種部位規則，之前也嘗試了各種配置器，以及更早的模型配置。最佳成員以某個幅度勝過其餘成員，而其中一部分幅度，只是因為在單一驗證期間的許多次抽樣中取出最佳者。結果本身不會標示這點：勝出停損規則的夏普比率、回撤與命中率，對該規則而言都計算正確；但只看到勝出者的讀者，會看到一個看似更好的策略，而看不到它是從多大集合中取出的最大值。這就是成員數量與領先者同樣重要、集合應固定而非只留下勝出者，以及保留樣本應對整個集合一次性使用、而非只對領先者使用的原因。

固定集合而非刪去其他成員，也會保留負面結果。使情況惡化的停損水準，也是關於此策略的發現；只有在落敗成員仍與勝出者並列於集合中時，才能看見這項發現。
<!-- ML4T_ZH_END c0009-p000 -->


In [6]:
if not include_preview:
    _holdout_set_name = research_name(CASE_STUDY_ID, "holdout-candidates", scope=POPULATION_NAME)
    holdout_candidates = CandidateSet.create(
        study,
        name=_holdout_set_name,
        members=[*baselines, *allocations, *risk_results],
        comparison_contract={"comparable_fields": ["label_artifact", "feature_artifacts", "cv"]},
        supersedes=candidate_set_supersedes(
            study,
            name=_holdout_set_name,
            declared=SUPERSEDES_CANDIDATE_SETS.get(_holdout_set_name),
        ),
    )
    selected = holdout_candidates.best_validation_sharpe()
    print(f"Frozen holdout candidate set: {holdout_candidates.hash}")
    print(f"Validation-selected backtest: {selected.hash}")
else:
    print("Preview risk results remain outside official populations and candidate sets.")

Frozen holdout candidate set: 07d118635d4d
Validation-selected backtest: 0c8d095371b6


## Key takeaways

- Risk variants descend from the same signal-plus-allocation selection cohort as cost variants,
  so a cost result and a risk result for one label describe the same parent strategy.
- Each comparison changes one declared position-risk rule. The identity audit enforces that,
  because two results differing in a stop and in something else cannot be subtracted and nothing
  in the output would say so.
- Cost-sensitivity rows are excluded from selection: including them would advance the strategy
  measured under the kindest cost assumption, not the best strategy.
- A stop reads the path inside a holding period, which no earlier stage does. Its measured
  benefit depends on what the bars carry and on the configured fill model.
- The final candidate set contains signal, allocation, and risk results across every label, and
  its size is the number of trials the eventual winner was drawn from.

The improvement a stop shows here is measured on validation, and the stop level was chosen by
looking at that same validation period. Whether it survives is the holdout's question, asked
once, and it is allowed to answer no.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 重點摘要

- 風險變體與成本變體，源自同一個訊號加配置候選群組，因此同一標籤的成本與風險結果都描述相同的母策略。
- 每項比較只改變一項預先宣告的部位風險規則。識別資訊稽核會強制檢查，因為若兩項結果同時在停損與其他因素上不同，就不能相減比較，而輸出不會指出此事。
- 成本敏感度資料列會排除於選擇之外：若納入，進入下一階段的會是在最寬鬆成本假設下衡量的策略，而非最佳策略。
- 停損會讀取持有期間內的價格路徑，之前任何階段都不會。測得的效益取決於價格柱所含資訊，以及設定的成交模型。
- 最終候選集合包含所有標籤的訊號、配置與風險結果；其大小就是最終勝出者被挑選時的試驗次數。

這裡顯示的停損改善是在驗證資料上衡量，而停損水準也依同一驗證期間的結果選出。改善能否延續，是保留樣本要一次性回答的問題；其答案可以是否定的。
<!-- ML4T_ZH_END c0011-p000 -->
